In [ ]:
import json
import logging

import joblib
import pandas as pd

from gensim import utils
from gensim.models.fasttext import FastText
from gensim.models.doc2vec import Doc2Vec, TaggedDocument
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from torch.utils.data import DataLoader, Dataset
from sentence_transformers import (
    InputExample,
    LoggingHandler,
    SentenceTransformer,
    losses,
)

from stemmer import Stemmer, tokenize

/home/fahmi/freelance/project-2023-amsearch/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
logging.basicConfig(
    format="%(asctime)s - %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
    level=logging.INFO,
    handlers=[LoggingHandler()],
)

In [3]:
## TRAINING PARAMS
STEMMING_AT_TRAIN = False

In [ ]:
def stem_sentence(stemmer: Stemmer, text: str) -> str:
    return " ".join([stemmer.stem_ams(t) for t in tokenize(text)])

## BoW & TF-IDF Model

In [4]:
class AMSTokenizer:
    def __init__(self, stm: Stemmer):
        self.stemmer = stm

    def __call__(self, doc):
        return [self.stemmer.stem_ams(word) for word in tokenize(doc)]

In [5]:
stemmer = Stemmer("../data/sundabaru1-vocab.txt")
amstokenizer = AMSTokenizer(stemmer)

In [6]:
df_corpus = pd.read_json("../data/cleaned/triplet.jsonl", lines=True)
df_corpus.head()

,query,positive,negative
0,Kumaha cara ngahontal kahayang dina kahirupan?,"Kahiji, urang kedah sabar sareng henteu janten...",Abdi ngadangu seueur warta ngeunaan jalma anu ...
1,Naon anu kedah dilakukeun lamun gering?,"Lamun gering, ulah rungsing sabab pikiran posi...",Masyarakat ayeuna seueur nganggur di kota nu g...
2,Kumaha cara nyieun amal?,"Ngawitan amal ti hal-hal leutik, sapertos ngab...",Jalma sering nyarita ngeunaan kaékonomian anu ...
3,Kumaha sangkan ngeterkeun diri ka Gusti?,"Mertahankeun ati, pariksa tindakan sorangan, s...","Saurang guru ngajarkeun pentingna ilmu, tapi k..."
4,Naon hartina jadi jalma leutik?,Jadi jalma leutik hartina ulah sombong sareng ...,"Dina pagelaran, anu katinggali gaduh prestasi ..."


In [7]:
train_corpus = df_corpus.values.ravel().tolist()
len(train_corpus), train_corpus[0]

(22476, 'Kumaha cara ngahontal kahayang dina kahirupan?')

In [8]:
vsm_model = CountVectorizer(tokenizer=amstokenizer if STEMMING_AT_TRAIN else None)
# vsm_model = TfidfVectorizer(tokenizer=amstokenizer if STEMMING_AT_TRAIN else None)

vsm_model.fit(train_corpus)

CountVectorizer()

In [9]:
joblib.dump(vsm_model, f"../tmp/ams-bow-stem.joblib")

['../tmp/ams-bow-stem.joblib']

## Doc2Vec & FastText

In [ ]:
class GensimTripletCorpusLoader:
    def __init__(self, corpus_path: str, stemmer: Stemmer=None):
        self.corpus_path = corpus_path
        self.stemmer = stemmer
        self.index = 0

    def __iter__(self):
        df_corpus = pd.read_json(self.corpus_path, lines=True)
        self.corpus = df_corpus.values.ravel().tolist()
        self.index = 0
        self.total = len(self.corpus)

        return self
        
    def __next__(self):
        if self.total == self.index:
            raise StopIteration
        
        line = self.corpus[self.index]
        if self.stemmer:
            line = stem_sentence(self.stemmer, line)

        self.index += 1
        return utils.simple_preprocess(line)

In [11]:
corpus_loader = GensimTripletCorpusLoader("../data/cleaned/triplet.jsonl")
# corpus_loader = GensimTripletCorpusLoader("../data/cleaned/triplet.jsonl", stemmer)

In [ ]:
# train_corpus = [TaggedDocument(x, [i]) for i, x in enumerate(corpus_loader)]
# gensim_model = Doc2Vec(vector_size=100, min_count=2, epochs=50)

In [12]:
train_corpus = list(corpus_loader)
gensim_model = FastText(vector_size=100, min_count=2, epochs=50)

2025-05-09 11:16:12 - FastText lifecycle event {'params': 'FastText<vocab=0, vector_size=100, alpha=0.025>', 'datetime': '2025-05-09T11:16:12.308581', 'gensim': '4.3.2', 'python': '3.11.12 (main, Apr  9 2025, 04:04:00) [Clang 20.1.0 ]', 'platform': 'Linux-5.15.167.4-microsoft-standard-WSL2-x86_64-with-glibc2.39', 'event': 'created'}


In [13]:
gensim_model.build_vocab(train_corpus)

2025-05-09 11:16:12 - collecting all words and their counts
2025-05-09 11:16:12 - PROGRESS: at sentence #0, processed 0 words, keeping 0 word types
2025-05-09 11:16:12 - PROGRESS: at sentence #10000, processed 122131 words, keeping 13226 word types
2025-05-09 11:16:12 - PROGRESS: at sentence #20000, processed 235832 words, keeping 19802 word types
2025-05-09 11:16:12 - collected 20960 word types from a corpus of 264095 raw words and 22476 sentences
2025-05-09 11:16:12 - Creating a fresh vocabulary
2025-05-09 11:16:13 - FastText lifecycle event {'msg': 'effective_min_count=2 retains 12065 unique words (57.56% of original 20960, drops 8895)', 'datetime': '2025-05-09T11:16:13.005896', 'gensim': '4.3.2', 'python': '3.11.12 (main, Apr  9 2025, 04:04:00) [Clang 20.1.0 ]', 'platform': 'Linux-5.15.167.4-microsoft-standard-WSL2-x86_64-with-glibc2.39', 'event': 'prepare_vocab'}
2025-05-09 11:16:13 - FastText lifecycle event {'msg': 'effective_min_count=2 leaves 255200 word corpus (96.63% of orig

In [14]:
gensim_model.train(train_corpus, total_examples=gensim_model.corpus_count, epochs=gensim_model.epochs)

2025-05-09 11:16:14 - FastText lifecycle event {'msg': 'training model with 3 workers on 12065 vocabulary and 100 features, using sg=0 hs=0 sample=0.001 negative=5 window=5 shrink_windows=True', 'datetime': '2025-05-09T11:16:14.424593', 'gensim': '4.3.2', 'python': '3.11.12 (main, Apr  9 2025, 04:04:00) [Clang 20.1.0 ]', 'platform': 'Linux-5.15.167.4-microsoft-standard-WSL2-x86_64-with-glibc2.39', 'event': 'train'}
2025-05-09 11:16:14 - EPOCH 0: training on 264095 raw words (216362 effective words) took 0.5s, 479345 effective words/s
2025-05-09 11:16:15 - EPOCH 1: training on 264095 raw words (216518 effective words) took 0.4s, 518258 effective words/s
2025-05-09 11:16:15 - EPOCH 2: training on 264095 raw words (216333 effective words) took 0.4s, 530181 effective words/s
2025-05-09 11:16:16 - EPOCH 3: training on 264095 raw words (216484 effective words) took 0.4s, 519121 effective words/s
2025-05-09 11:16:16 - EPOCH 4: training on 264095 raw words (216604 effective words) took 0.5s, 4

(10820260, 13204750)

In [15]:
# gensim_model.save("../tmp/ams-doc2vec-stem.model")
gensim_model.save("../tmp/ams-fasttext-stem.model")

2025-05-09 11:16:39 - FastText lifecycle event {'fname_or_handle': '../tmp/ams-fasttext-stem.model', 'separately': 'None', 'sep_limit': 10485760, 'ignore': frozenset(), 'datetime': '2025-05-09T11:16:39.503051', 'gensim': '4.3.2', 'python': '3.11.12 (main, Apr  9 2025, 04:04:00) [Clang 20.1.0 ]', 'platform': 'Linux-5.15.167.4-microsoft-standard-WSL2-x86_64-with-glibc2.39', 'event': 'saving'}
2025-05-09 11:16:39 - storing np array 'vectors_ngrams' to ../tmp/ams-fasttext-stem.model.wv.vectors_ngrams.npy
2025-05-09 11:16:40 - not storing attribute vectors
2025-05-09 11:16:40 - not storing attribute buckets_word
2025-05-09 11:16:40 - not storing attribute cum_table
2025-05-09 11:16:40 - saved ../tmp/ams-fasttext-stem.model


In [16]:
if isinstance(gensim_model, Doc2Vec):
    vec = gensim_model.infer_vector(['henteu', 'sampurna', 'pamulang', 'samemehna'])
else:
    vec = gensim_model.wv.get_sentence_vector('henteu sampurna pamulang samemehna')

print(vec)

[ 0.00616358 -0.01715619 -0.00262772  0.01694225 -0.01925823 -0.01249245
  0.02322457  0.00925869 -0.01779497  0.03399766  0.02925516 -0.04467531
  0.00101425 -0.00665259  0.00110258  0.0325543   0.0210901   0.04633536
 -0.02268081 -0.01349142 -0.05788888  0.00737747  0.03225493  0.07551904
 -0.02027582 -0.0675667  -0.05491181  0.00808581 -0.07999872 -0.0389426
  0.00973993 -0.0132723   0.08658468  0.04207335  0.03754553  0.02538137
  0.03352181  0.03861135 -0.01507373 -0.04042877 -0.03445201 -0.04658359
 -0.01974014  0.01050041 -0.00412998  0.03439814 -0.06764229  0.02017625
  0.00671273  0.0069877   0.00690931  0.05645541 -0.03185806  0.009387
 -0.02856121 -0.04459525 -0.0166617  -0.01725537 -0.02660958  0.0196685
 -0.00380388 -0.01640305  0.0428814  -0.01629054  0.00410026 -0.01231399
  0.01462355 -0.03592745  0.0100183  -0.05206258 -0.00952881 -0.04449329
 -0.047554   -0.05562646 -0.04125122  0.00268497 -0.0060323  -0.01005628
 -0.02302903  0.05402196  0.0195016  -0.03963586 -0.047

## Dense Model

In [ ]:
class TripletDataset(Dataset):
    def __init__(self, dataset_path: str, stemmer: Stemmer = None):
        self.stemmer = stemmer
        with open(dataset_path, "r") as f:
            self.dataset = [json.loads(x) for x in f]

    def __getitem__(self, idx):
        item = self.dataset[idx]

        if self.stemmer:
            return InputExample(
                texts=[
                    stem_sentence(self.stemmer, item["query"]),
                    stem_sentence(self.stemmer, item["positive"]),
                    stem_sentence(self.stemmer, item["negative"]),
                ]
            )

        return InputExample(texts=[item["query"], item["positive"], item["negative"]])

    def __len__(self):
        return len(self.dataset)

In [ ]:
train_dataset = TripletDataset("../data/triplet/triplet.jsonl")
# train_dataset = TripletDataset("../data/triplet/triplet.jsonl", stemmer=stemmer)

train_dataloader = DataLoader(train_dataset, shuffle=True, batch_size=64)

In [ ]:
model = SentenceTransformer("sentence-transformers/msmarco-distilbert-cos-v5")
model.max_seq_length = 512

In [ ]:
train_loss = losses.MultipleNegativesRankingLoss(model=model)

In [ ]:
model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    use_amp=True,
    epochs=10,
    warmup_steps=10000,
    optimizer_params={"lr": 2e-5},
)

In [ ]:
model.save(f"../tmp/ams-dense")